# Notebook 2: Advanced Preprocessing
This notebook transforms our raw Greater London collision dataset (from Notebook 1) into a clean, model-ready feature matrix for PyTorch.

We go beyond the baseline academic brief by applying:
- **KNN Imputation** for intelligent missing-value estimation
- **Target Encoding** for high-cardinality categorical features
- **SMOTE-NC** for synthesising realistic minority-class (Fatal) training samples

Every decision is documented and justified from a Data Science perspective.

## Step 1: Imports, Seeds, and Data Reload
We reload the London dataset by re-executing the exact same pipeline from Notebook 1. This guarantees full reproducibility — any reviewer can run this notebook from a fresh kernel without depending on hidden state.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import joblib

warnings.filterwarnings('ignore')
np.random.seed(42)

# Reload data using the exact Notebook 1 pipeline
collisions = pd.read_csv('../data/dft-road-casualty-statistics-collision-2024.csv', low_memory=False)
casualties = pd.read_csv('../data/dft-road-casualty-statistics-casualty-2024.csv', low_memory=False)
vehicles = pd.read_csv('../data/dft-road-casualty-statistics-vehicle-2024.csv', low_memory=False)

# Feature Aggregation (from Notebook 1 Step 3)
casualties_agg = casualties.groupby('collision_index').agg(
    total_casualties=('casualty_reference', 'count'),
    min_casualty_age=('age_of_casualty', 'min'),
    max_casualty_age=('age_of_casualty', 'max')
).reset_index()

vehicles_agg = vehicles.groupby('collision_index').agg(
    total_vehicles_involved=('vehicle_reference', 'count'),
    min_driver_age=('age_of_driver', 'min'),
    max_driver_age=('age_of_driver', 'max')
).reset_index()

# Merge and filter to London (ONS E09 codes)
df = collisions.merge(casualties_agg, on='collision_index', how='left')
df = df.merge(vehicles_agg, on='collision_index', how='left')
london_df = df[df['local_authority_ons_district'].fillna('').str.startswith('E09')].copy()

print(f"London dataset reloaded: {london_df.shape[0]:,} rows, {london_df.shape[1]} columns")

London dataset reloaded: 20,967 rows, 50 columns


## Step 1b: Select Strict Prediction-Time Features
We now surgically remove every column that would NOT be available to IntelliSys's sensor network at prediction time. 

**Columns we DROP and why:**
| Category | Columns | Reason |
|----------|---------|--------|
| **Identifiers** | `collision_index`, `collision_ref_no`, `collision_year` | Administrative IDs with zero predictive power |
| **GPS / Location codes** | `location_easting_osgr`, `location_northing_osgr`, `longitude`, `latitude`, `lsoa_of_accident_location`, `local_authority_*` | High-cardinality location data that would cause overfitting; location risk is better captured by road type and speed limit |
| **Police / Administrative** | `police_force`, `did_police_officer_attend_scene_of_accident`, `trunk_road_flag` | Post-crash administrative data not available at prediction time |
| **Target Leakage** | `enhanced_severity_collision`, `collision_injury_based`, `collision_adjusted_severity_serious`, `collision_adjusted_severity_slight` | These are derived directly from the target variable — using them would give the model the answer key |
| **Redundant / Historic** | `junction_detail_historic`, `pedestrian_crossing_human_control_historic`, `pedestrian_crossing_physical_facilities_historic`, `carriageway_hazards_historic` | Superseded by their modern equivalents already in the dataset |
| **Raw Date/Road Numbers** | `date`, `first_road_number`, `second_road_number`, `second_road_class` | Date is captured by `day_of_week`; road numbers are high-cardinality IDs |

In [2]:
# Columns to explicitly drop
drop_cols = [
    # Identifiers
    'collision_index', 'collision_ref_no', 'collision_year',
    # GPS / Location codes
    'location_easting_osgr', 'location_northing_osgr',
    'longitude', 'latitude', 'lsoa_of_accident_location',
    'local_authority_district', 'local_authority_ons_district',
    'local_authority_highway', 'local_authority_highway_current',
    # Police / Administrative (not available at prediction time)
    'police_force', 'did_police_officer_attend_scene_of_accident', 'trunk_road_flag',
    # Target Leakage (derived from target variable)
    'enhanced_severity_collision', 'collision_injury_based',
    'collision_adjusted_severity_serious', 'collision_adjusted_severity_slight',
    # Redundant historic columns
    'junction_detail_historic', 'pedestrian_crossing_human_control_historic',
    'pedestrian_crossing_physical_facilities_historic', 'carriageway_hazards_historic',
    # High-cardinality road numbers and raw date
    'date', 'first_road_number', 'second_road_number', 'second_road_class',
]

# Separate target from features
target_col = 'collision_severity'

# Drop columns safely
existing_drop = [c for c in drop_cols if c in london_df.columns]
model_df = london_df.drop(columns=existing_drop).copy()

# Extract the time column as hour-of-day (numeric) before dropping the string
model_df['hour'] = pd.to_datetime(model_df['time'], format='%H:%M', errors='coerce').dt.hour
model_df.drop(columns=['time'], inplace=True)

print(f"Dropped {len(existing_drop)} administrative/leakage columns.")
print(f"Remaining features: {model_df.shape[1] - 1} + 1 target")
print(f"\nFinal columns:\n{list(model_df.columns)}")

Dropped 27 administrative/leakage columns.
Remaining features: 22 + 1 target

Final columns:
['collision_severity', 'number_of_vehicles', 'number_of_casualties', 'day_of_week', 'first_road_class', 'road_type', 'speed_limit', 'junction_detail', 'junction_control', 'pedestrian_crossing', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'special_conditions_at_site', 'carriageway_hazards', 'urban_or_rural_area', 'total_casualties', 'min_casualty_age', 'max_casualty_age', 'total_vehicles_involved', 'min_driver_age', 'max_driver_age', 'hour']


## Step 2: Remap the Target Variable
PyTorch's `nn.CrossEntropyLoss` strictly requires class labels to start from `0`. Our raw `collision_severity` uses the STATS19 encoding: `1=Fatal, 2=Serious, 3=Slight`.

We remap as follows:
- `1` (Fatal) → `0`
- `2` (Serious) → `1`
- `3` (Slight) → `2`

This is a critical bookkeeping step — if we forget this, PyTorch will throw a silent index-out-of-bounds error or, worse, train successfully but on corrupted labels. We print the class distribution to confirm the extreme imbalance for IntelliSys's awareness.

In [3]:
# Remap target: Fatal=0, Serious=1, Slight=2
model_df['target'] = model_df['collision_severity'].map({1: 0, 2: 1, 3: 2})

# Drop the original collision_severity column to avoid confusion
model_df.drop(columns=['collision_severity'], inplace=True)

# Verify the mapping and show class distribution
class_counts = model_df['target'].value_counts().sort_index()
class_labels = {0: 'Fatal', 1: 'Serious', 2: 'Slight'}
print("Target Variable Distribution:")
print("-" * 40)
for cls, count in class_counts.items():
    pct = count / len(model_df) * 100
    print(f"  {cls} ({class_labels[cls]:>7s}): {count:>6,} samples ({pct:.1f}%)")
print(f"\nTotal: {len(model_df):,} rows, {model_df.shape[1]} columns (including target)")

Target Variable Distribution:
----------------------------------------
  0 (  Fatal):    108 samples (0.5%)
  1 (Serious):  3,493 samples (16.7%)
  2 ( Slight): 17,366 samples (82.8%)

Total: 20,967 rows, 23 columns (including target)


## Step 3: Hybrid Imputation Strategy (Sentinel Value Handling)

Our diagnostic reveals that STATS19 uses `-1` as a sentinel code meaning "data missing or unknown." We found sentinel values in these features:

| Feature | `-1` Count | % | Strategy |
|---------|-----------|---|----------|
| `min_driver_age` | 8,631 | 41.2% | **DROP column** — too much missing data to reliably impute |
| `junction_control` | 4,080 | 19.5% | Keep `-1` as "Unknown" category |
| `junction_detail` | 2,256 | 10.8% | Keep `-1` as "Unknown" category |
| `min_casualty_age` | 881 | 4.2% | KNN Imputation (real number) |
| `max_casualty_age` | 498 | 2.4% | KNN Imputation (real number) |
| `max_driver_age` | 1,637 | 7.8% | KNN Imputation (real number) |
| `special_conditions_at_site` | 181 | 0.9% | Keep `-1` as "Unknown" category |
| `pedestrian_crossing` | 6 | 0.0% | Keep `-1` as "Unknown" category |
| `carriageway_hazards` | 6 | 0.0% | Keep `-1` as "Unknown" category |

**Design Decisions:**
1. **`min_driver_age` is dropped** because 41.2% missing data makes any imputation unreliable — we would be guessing nearly half the values, which injects noise rather than signal.
2. **Categorical `-1`s are kept** as a distinct "Unknown" category because the fact that the officer couldn't determine the junction type or conditions is itself potentially informative (MNAR — Missing Not At Random).
3. **Numeric age features** are imputed via **KNN (k=5)** using the non-sentinel features as neighbors, because a driver age of `-1` is genuinely nonsensical.

In [4]:
from sklearn.impute import KNNImputer

# 1. DROP min_driver_age (41.2% missing — too unreliable)
model_df.drop(columns=['min_driver_age'], inplace=True)
print("Dropped 'min_driver_age' (41.2% sentinel values — too unreliable to impute)")

# 2. CATEGORICAL FEATURES: Keep -1 as a legitimate 'Unknown' category
# (No code action needed — they stay as-is and will be encoded later)
categorical_sentinel_cols = [
    'junction_detail', 'junction_control', 'pedestrian_crossing',
    'special_conditions_at_site', 'carriageway_hazards'
]
for col in categorical_sentinel_cols:
    count = (model_df[col] == -1).sum()
    if count > 0:
        print(f"  '{col}': keeping {count:,} sentinel values as 'Unknown' category")

# 3. NUMERIC FEATURES: KNN Impute ages where -1 exists
numeric_impute_cols = ['min_casualty_age', 'max_casualty_age', 'max_driver_age']

# Replace -1 with NaN so KNN can detect and impute them
for col in numeric_impute_cols:
    sentinel_count = (model_df[col] == -1).sum()
    model_df.loc[model_df[col] == -1, col] = np.nan
    print(f"  '{col}': marked {sentinel_count:,} sentinel values for KNN imputation")

# Run KNN Imputer (k=5) on numeric columns only
imputer = KNNImputer(n_neighbors=5)
model_df[numeric_impute_cols] = imputer.fit_transform(model_df[numeric_impute_cols])

# Verify: confirm zero NaN values remain
remaining_nan = model_df[numeric_impute_cols].isnull().sum().sum()
print(f"\nKNN Imputation complete. Remaining NaN in imputed columns: {remaining_nan}")
print(f"Dataset shape: {model_df.shape[0]:,} rows, {model_df.shape[1]} columns")

Dropped 'min_driver_age' (41.2% sentinel values — too unreliable to impute)
  'junction_detail': keeping 2,256 sentinel values as 'Unknown' category
  'junction_control': keeping 4,080 sentinel values as 'Unknown' category
  'pedestrian_crossing': keeping 6 sentinel values as 'Unknown' category
  'special_conditions_at_site': keeping 181 sentinel values as 'Unknown' category
  'carriageway_hazards': keeping 6 sentinel values as 'Unknown' category
  'min_casualty_age': marked 881 sentinel values for KNN imputation
  'max_casualty_age': marked 498 sentinel values for KNN imputation
  'max_driver_age': marked 1,637 sentinel values for KNN imputation



KNN Imputation complete. Remaining NaN in imputed columns: 0
Dataset shape: 20,967 rows, 22 columns


## Step 4: Stratified Train / Validation / Test Split
**Critical ordering decision:** We MUST split the data BEFORE applying Target Encoding (Step 5). 

Target Encoding works by replacing each category with the average severity of crashes in that category. If we computed those averages on the entire dataset (including test data), then the test set's target values would silently leak into the training features — the model would have seen a "preview" of the test answers during training.

By splitting first, we guarantee the encoding is computed **only from training data**, and the validation and test sets are transformed using those training-derived statistics.

We use a **70% / 15% / 15%** stratified split, ensuring all three sets maintain the same Fatal/Serious/Slight proportions as the original dataset.

In [5]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = model_df.drop(columns=['target'])
y = model_df['target']

# 70% Train, 30% Temp (stratified)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Split Temp evenly into 15% Val, 15% Test (stratified)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Verify stratification preserved the class ratios
print("Split Sizes:")
print(f"  Train: {X_train.shape[0]:>6,} rows ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  Val:   {X_val.shape[0]:>6,} rows ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"  Test:  {X_test.shape[0]:>6,} rows ({X_test.shape[0]/len(X)*100:.1f}%)")

print("\nClass Distribution per Split:")
for name, ys in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    counts = ys.value_counts().sort_index()
    pcts = (counts / len(ys) * 100).round(1)
    labels = {0: 'Fatal', 1: 'Serious', 2: 'Slight'}
    dist = ', '.join([f"{labels[i]}={pcts[i]}%" for i in sorted(counts.index)])
    print(f"  {name:5s}: {dist}")

Split Sizes:
  Train: 14,676 rows (70.0%)
  Val:    3,145 rows (15.0%)
  Test:   3,146 rows (15.0%)

Class Distribution per Split:
  Train: Fatal=0.5%, Serious=16.7%, Slight=82.8%
  Val  : Fatal=0.5%, Serious=16.7%, Slight=82.8%
  Test : Fatal=0.5%, Serious=16.7%, Slight=82.8%


## Step 5: Target Encoding (Fitted on Training Data Only)
Instead of One-Hot Encoding (which would explode our 21 features into 80+ sparse binary columns), we use **Target Encoding** for categorical features.

**How it works:** For each category value (e.g., `weather_conditions = 2` meaning "Raining"), we replace it with the average target value across all training-set crashes where it was raining. If rainy crashes average a severity of `1.85` (closer to 2=Slight), the model directly receives `1.85` — an extremely dense, informative numeric signal.

**Why this is superior to One-Hot:**
- Reduces dimensionality instead of inflating it
- Directly encodes the relationship between the category and severity
- Works perfectly with the MLP because every input is already a pure number

**Leakage protection:** We use `category_encoders.TargetEncoder` with built-in **leave-one-out smoothing**, which prevents any single row from influencing its own encoding. The encoder is fitted strictly on training data, then applied to val/test.

In [6]:
import category_encoders as ce

# Identify categorical features (those with relatively few unique values that are codes, not real numbers)
categorical_cols = [
    'day_of_week', 'first_road_class', 'road_type', 'speed_limit',
    'junction_detail', 'junction_control', 'pedestrian_crossing',
    'light_conditions', 'weather_conditions', 'road_surface_conditions',
    'special_conditions_at_site', 'carriageway_hazards', 'urban_or_rural_area'
]

# Identify numeric features (true continuous values)
numeric_cols = [col for col in X_train.columns if col not in categorical_cols]

print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")
print(f"Numeric features ({len(numeric_cols)}): {numeric_cols}")

# Fit Target Encoder on TRAINING DATA ONLY
target_encoder = ce.TargetEncoder(cols=categorical_cols, smoothing=1.0)
X_train_encoded = target_encoder.fit_transform(X_train, y_train)
X_val_encoded = target_encoder.transform(X_val)
X_test_encoded = target_encoder.transform(X_test)

print(f"\nEncoded shapes — Train: {X_train_encoded.shape}, Val: {X_val_encoded.shape}, Test: {X_test_encoded.shape}")
print(f"All features are now pure numeric: {X_train_encoded.dtypes.unique()}")

Categorical features (13): ['day_of_week', 'first_road_class', 'road_type', 'speed_limit', 'junction_detail', 'junction_control', 'pedestrian_crossing', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'special_conditions_at_site', 'carriageway_hazards', 'urban_or_rural_area']
Numeric features (8): ['number_of_vehicles', 'number_of_casualties', 'total_casualties', 'min_casualty_age', 'max_casualty_age', 'total_vehicles_involved', 'max_driver_age', 'hour']



Encoded shapes — Train: (14676, 21), Val: (3145, 21), Test: (3146, 21)
All features are now pure numeric: [dtype('int64') dtype('float64') dtype('int32')]


## Step 6: Compute Class Weights for Focal Loss
Instead of SMOTE (which risks generating unrealistic synthetic crashes from only 76 Fatal samples), we adopt the research-backed **Focal Loss + Class Weights** hybrid strategy.

Here, we compute the inverse-frequency class weights using scikit-learn. These weights will be passed directly to our custom Focal Loss function in the training loop (Notebook 4). The maths is simple: rarer classes get exponentially higher weights, forcing the model to treat Fatal misclassifications as catastrophic errors.

In [7]:
from sklearn.utils.class_weight import compute_class_weight

# Compute inverse-frequency weights from training labels only
classes = np.array([0, 1, 2])
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train.values)
class_weight_dict = {i: w for i, w in zip(classes, class_weights)}

print("Class Weights (computed from training set only):")
labels = {0: 'Fatal', 1: 'Serious', 2: 'Slight'}
for cls, weight in class_weight_dict.items():
    print(f"  {cls} ({labels[cls]:>7s}): weight = {weight:.2f}")

print(f"\nInterpretation: a Fatal misclassification is penalised {class_weights[0]/class_weights[2]:.0f}x harder than a Slight one.")

Class Weights (computed from training set only):
  0 (  Fatal): weight = 64.37
  1 (Serious): weight = 2.00
  2 ( Slight): weight = 0.40

Interpretation: a Fatal misclassification is penalised 160x harder than a Slight one.


## Step 7: Standard Scale Features and Save Artefacts
Neural networks train most efficiently when all input features have a similar numeric range. `StandardScaler` transforms each column to have a mean of 0 and a standard deviation of 1.

**Critical:** We fit the scaler on training data only, then apply the same transformation to val and test. This prevents any information from the val/test distributions leaking into training.

We save all processed artefacts (`X_train`, `y_train`, `X_val`, `y_val`, `X_test`, `y_test`, `scaler`, `class_weights`, `feature_names`) to disk using `joblib` so that Notebooks 3–6 can load them instantly without re-running preprocessing.

In [8]:
from sklearn.preprocessing import StandardScaler
import torch

# Fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_val_scaled = scaler.transform(X_val_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

print(f"Scaled Train — mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}")
print(f"Scaled Val   — mean: {X_val_scaled.mean():.4f}, std: {X_val_scaled.std():.4f}")
print(f"Scaled Test  — mean: {X_test_scaled.mean():.4f}, std: {X_test_scaled.std():.4f}")

# Save all artefacts for downstream notebooks
os.makedirs('../outputs/models', exist_ok=True)

artefacts = {
    'X_train': X_train_scaled.astype(np.float32),
    'X_val': X_val_scaled.astype(np.float32),
    'X_test': X_test_scaled.astype(np.float32),
    'y_train': y_train.values.astype(np.int64),
    'y_val': y_val.values.astype(np.int64),
    'y_test': y_test.values.astype(np.int64),
    'class_weights': class_weights.astype(np.float32),
    'feature_names': list(X_train_encoded.columns),
    'scaler': scaler,
}

joblib.dump(artefacts, '../outputs/models/preprocessed_data.joblib')

print(f"\nSaved {len(artefacts)} artefacts to outputs/models/preprocessed_data.joblib")
print(f"Feature names ({len(artefacts['feature_names'])}): {artefacts['feature_names']}")
print(f"\n--- Phase 3 Preprocessing Complete ---")

Scaled Train — mean: 0.0000, std: 1.0000
Scaled Val   — mean: -0.0084, std: 0.9965
Scaled Test  — mean: -0.0153, std: 1.0207

Saved 9 artefacts to outputs/models/preprocessed_data.joblib
Feature names (21): ['number_of_vehicles', 'number_of_casualties', 'day_of_week', 'first_road_class', 'road_type', 'speed_limit', 'junction_detail', 'junction_control', 'pedestrian_crossing', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'special_conditions_at_site', 'carriageway_hazards', 'urban_or_rural_area', 'total_casualties', 'min_casualty_age', 'max_casualty_age', 'total_vehicles_involved', 'max_driver_age', 'hour']

--- Phase 3 Preprocessing Complete ---
